# Figures for "Voluntary social confinement in disease spreading: multiple waves and optimal timing of non-pharmaceutical interventions" (SIRC model)

This notebook regenerates Figs. 2–7 of the manuscript from scratch, and the three supplementary figures S1–S3** (Sections 9, 9b and 10). Fig. 6 is built from pre-computed data (`data_fig_6/*.csv`, produced with the Julia script `compute_fig6_data.jl`), because it requires $\sim 4\cdot10^{5}$ integrations of the model; the cell that recomputes it from scratch is kept, but it is switched off by default (see Section 6).

**Model.** SIRC compartments (confined, susceptible, infected, recovered) with confinement rate $\alpha$ (spontaneous, Sec. 3.1) or $\eta\,\rho_i$ (prevalence-driven, Sec. 3.2), return rate $\omega$, infection rate $\beta$ and recovery rate $\gamma$. Campaigns of finite duration (Sec. 4) switch the confinement on during $t_{ini}\le t\le\min(t_{ini}+\tau,\,T)$.

**Reproducibility notes**

* All ODEs are integrated with `scipy.integrate.solve_ivp` (LSODA, `rtol=1e-7`; final sizes are accurate to ~$10^{-7}$). The model and the integration helpers are in Section 2 (about 40 lines); Section 2b checks the solver against the exact final size of the classical SIR model.
* Parameters are those of the manuscript ($\beta=0.2$, $\gamma=0.1$, $\omega=0.025$, $\rho_i(0)=10^{-6}$, $\rho_s(0)=1-10^{-6}$) unless stated otherwise in each section. The campaign figures (6, 7) integrate up to $t=3000$; the final size is converged to $\sim10^{-3}$ at that time (it keeps creeping up by $\lesssim 5\cdot10^{-4}$ up to $t=2\cdot10^4$).
* Every figure is built **at its final size on the page** (17 cm = full text width), so the font sizes in the files are the font sizes in print. Change the constants of Section 1 to adapt to a different template.
* Typography is LaTeX-like (Computer Modern text and maths through matplotlib's built-in `mathtext`), so no TeX installation is needed. Axis labels, legends and annotations are typeset in math mode.
* Figures are written to `../figures/` (the `figures/` folder of the package, next to `code/`) as vector PDF (fonts embedded) and 300-dpi PNG.
* Run time: a few minutes with the default settings. Fig. 6 is read from `data_fig_6/` (full resolution, 1 unit steps in $\tau$ and $t_{ini}$). Recomputing it in Python is optional (`RECOMPUTE_FIG6 = True` in Section 6) and takes **hours** at full resolution (roughly 15–30 minutes with the coarse default `STEP5 = 3`, depending on the machine).

Requirements: `numpy`, `scipy`, `matplotlib >= 3.7`.


## 1. Setup and common style

In [ ]:
# ---------------------------------------------------------------------------
# 1. Setup and common style
# ---------------------------------------------------------------------------
import os, time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FixedLocator, FormatStrFormatter

OUTDIR = "../figures"      # the figures/ folder of the package (next to code/)
os.makedirs(OUTDIR, exist_ok=True)

CM = 1 / 2.54            # inches per cm
W_FULL = 17.0            # full text width of the journal [cm] (check with \the\textwidth)

# Font sizes in points -- identical in every figure, main panels and insets alike.
FS_TICK, FS_LABEL, FS_LEGEND, FS_ANNOT, FS_PANEL = 9, 15, 10, 10, 14
FS_TITLE_DYN = 12      # column titles of the dynamics panels (Figs. 5b, 7b)

mpl.rcParams.update({
    # LaTeX-like typography without requiring a TeX installation: Computer Modern for text and maths
    "font.family": "serif",
    "font.serif": ["cmr10", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,            # cmr10 has no Unicode minus sign
    "axes.formatter.use_mathtext": True,
    "pdf.fonttype": 42, "ps.fonttype": 42,          # embed TrueType fonts
    "font.size": FS_TICK,
    "axes.labelsize": FS_LABEL, "axes.titlesize": FS_LABEL,
    "xtick.labelsize": FS_TICK, "ytick.labelsize": FS_TICK,
    "legend.fontsize": FS_LEGEND, "legend.frameon": True, "legend.framealpha": 1.0,
    "legend.edgecolor": "0.3", "legend.fancybox": False, "legend.borderpad": 0.35,
    "legend.labelspacing": 0.25, "legend.handlelength": 1.8, "legend.handletextpad": 0.5,
    "axes.linewidth": 0.8, "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "xtick.major.size": 3, "ytick.major.size": 3, "xtick.minor.size": 1.8, "ytick.minor.size": 1.8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "lines.linewidth": 1.6, "savefig.dpi": 300,
})

def new_fig(w_cm, h_cm):
    fig = plt.figure(figsize=(w_cm * CM, h_cm * CM))
    fig.wcm, fig.hcm = w_cm, h_cm
    return fig

def axes_cm(fig, left, bottom, w, h, **kw):
    """Axes placed with absolute dimensions in cm (origin: bottom-left corner of the figure)."""
    return fig.add_axes([left / fig.wcm, bottom / fig.hcm, w / fig.wcm, h / fig.hcm], **kw)

def panel_label(ax, letter, dx=0.0, dy=0.12):
    """Bold '(a)' label just above the top-left corner of an axes (offsets in cm)."""
    ax.text(dx / (ax.get_position().width * ax.figure.wcm), 1 + dy / (ax.get_position().height * ax.figure.hcm),
            rf"$\mathbf{{({letter})}}$", transform=ax.transAxes, fontsize=FS_PANEL, ha="left", va="bottom")

def panel_label_inside(ax, letter, color="black", fx=0.03, fy=0.97):
    """Bold '(a)' label inside the top-left corner of an axes (axes-fraction coordinates)."""
    ax.text(fx, fy, rf"$\mathbf{{({letter})}}$", transform=ax.transAxes, fontsize=FS_PANEL,
            ha="left", va="top", color=color, zorder=10)

def save_fig(fig, name):
    fig.savefig(f"{OUTDIR}/{name}.pdf")
    fig.savefig(f"{OUTDIR}/{name}.png", dpi=300)
    print(f"{name}: {fig.wcm:.1f} x {fig.hcm:.1f} cm")

# Line styles: analytical approximation = thick, semi-transparent solid; numerical integration = dotted.
def plot_pair(ax, x, y_ana, y_num, color, label=None, lw_ana=2.6, lw_num=1.7, **kw):
    ax.plot(x, y_ana, color=color, lw=lw_ana, alpha=0.5, solid_capstyle="butt", label=label, **kw)
    ax.plot(x, y_num, color=color, lw=lw_num, ls=(0, (1.0, 1.4)), dash_capstyle="round")

import sys, scipy
print("python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__, "| matplotlib", mpl.__version__)


## 2. The SIRC model and how we integrate it

In [ ]:
# ---------------------------------------------------------------------------
# 2. The SIRC model and how we integrate it
# ---------------------------------------------------------------------------
# Compartments (fractions of the population): confined (c), susceptible (s), infected (i), recovered (r).
# Susceptible agents become confined at a rate Gamma_{s->c}, return at rate omega, get infected at
# rate beta * rho_i, and infected agents recover at rate gamma:
#
#   d rho_c / dt =  Gamma * rho_s - omega * rho_c
#   d rho_s / dt =  omega * rho_c - Gamma * rho_s - beta * rho_s * rho_i
#   d rho_i / dt =  beta * rho_s * rho_i - gamma * rho_i
#   d rho_r / dt =  gamma * rho_i
#
# Two choices for the confinement rate Gamma_{s->c}:
#   mode = "alpha":  Gamma = alpha            (spontaneous confinement, constant rate)
#   mode = "eta":    Gamma = eta * rho_i      (prevalence-driven confinement)
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

PARAMS = dict(beta=0.2, gamma=0.1, omega=0.025)      # default parameters (those of the manuscript)
I0 = 1e-6                                            # initial infected fraction; the rest is susceptible
# Integration tolerances. rho_i can become extremely small (1e-30 or less) between outbreaks, so its absolute
# tolerance is tiny: this keeps rho_i positive (a slightly negative rho_i would blow up when R_eff > 1).
RTOL, ATOL = 1e-7, [1e-10, 1e-10, 1e-60, 1e-10]

def sirc_rhs(t, y, mode, a, omega, beta, gamma):
    """Right-hand side of the SIRC equations; `a` is alpha (mode 'alpha') or eta (mode 'eta')."""
    rho_c, rho_s, rho_i, rho_r = y
    gamma_sc = a if mode == "alpha" else a * rho_i   # confinement rate Gamma_{s->c}
    return [gamma_sc * rho_s - omega * rho_c,
            omega * rho_c - gamma_sc * rho_s - beta * rho_s * rho_i,
            beta * rho_s * rho_i - gamma * rho_i,
            gamma * rho_i]

def integrate(mode, a, t_on, t_off, t_end, t_eval=None, **par):
    """Integrate the model from t = 0 to t_end.

    The confinement parameter `a` is switched on at t_on and off at t_off (it is zero outside this
    window); for permanent confinement use t_on=0, t_off=np.inf. The integration is split into
    intervals where `a` is constant, and each one is solved with scipy's solve_ivp (LSODA).
    `par` may override beta, gamma and omega. Returns the times and the states (columns: c, s, i, r);
    only t_end is returned unless t_eval is given.
    """
    p = {**PARAMS, **par}
    y = np.array([0.0, 1.0 - I0, I0, 0.0])                       # initial condition
    edges = sorted({0.0, float(t_end), min(max(t_on, 0.0), t_end), min(max(t_off, 0.0), t_end)})
    times, states = [np.array([0.0])], [y[None, :]]
    for t_start, t_stop in zip(edges[:-1], edges[1:]):
        active = (t_on <= t_start) and (t_stop <= t_off)          # is the campaign on in this interval?
        out = [t_stop] if t_eval is None else np.append(t_eval[(t_eval > t_start) & (t_eval < t_stop)], t_stop)
        sol = solve_ivp(sirc_rhs, (t_start, t_stop), y, method="LSODA", t_eval=out, rtol=RTOL, atol=ATOL,
                        args=(mode, a if active else 0.0, p["omega"], p["beta"], p["gamma"]))
        if not sol.success:
            raise RuntimeError(f"integration failed: {sol.message}")
        y = sol.y[:, -1]
        times.append(sol.t); states.append(sol.y.T)
    return np.concatenate(times), np.vstack(states)

def final_size(mode, a, t_on=0.0, t_off=np.inf, t_end=2000.0, **par):
    """Fraction of ever-infected agents rho_r(t_end)."""
    return integrate(mode, a, t_on, t_off, t_end, **par)[1][-1, 3]

def time_series(mode, a, t_on, t_off, t_eval, **par):
    """States (columns: c, s, i, r) at the times t_eval (a sorted numpy array)."""
    t, Y = integrate(mode, a, t_on, t_off, t_eval[-1], t_eval=t_eval, **par)
    return Y[np.isin(t, t_eval)]

def campaign_final_size(mode, a, t_ini, tau, T, t_end=3000.0, **par):
    """Final size for a campaign of strength `a` that starts at t_ini and lasts tau (it cannot go past T)."""
    return final_size(mode, a, t_on=t_ini, t_off=min(t_ini + tau, T), t_end=t_end, **par)

# ---- quasi-steady analytical approximations (manuscript Eqs. 7 and 22) --------------------------------
def rho_inf_const(beta, gamma, alpha, omega, rho_s0=1 - I0):
    """Eq. (7): final size for constant confinement rate, from the effective SIR model."""
    beta_eff = beta / (1 + alpha / omega)
    equation = lambda x: x - (1 - rho_s0 * np.exp(-beta_eff / gamma * x))
    return brentq(equation, 0.0, 1.0, xtol=1e-14)

def rho_inf_eta(beta, gamma, omega, eta):
    """Eq. (22): final size for prevalence-driven confinement (NaN if the equation has no root)."""
    equation = lambda x: x - (1 + omega / eta - gamma / beta) * (1 - (1 - x) ** (eta * gamma / (beta * omega)))
    try:
        return brentq(equation, 1e-10, 1 - 1e-10, xtol=1e-14)
    except ValueError:
        return np.nan

def analytic_and_numeric(x_values, analytic, numeric):
    """Evaluate an analytical approximation and the numerical integration for each value in x_values."""
    return np.array([analytic(x) for x in x_values]), np.array([numeric(x) for x in x_values])


## 2b. Sanity check of the solver

In [ ]:
# ---------------------------------------------------------------------------
# 2b. Sanity check against an exact result
# ---------------------------------------------------------------------------
# Without confinement the model reduces to the classical SIR model, whose final size solves
#   rho_r = 1 - (1 - I0) * exp(-R0 * rho_r),   with R0 = beta / gamma.
R0 = PARAMS["beta"] / PARAMS["gamma"]
exact = brentq(lambda x: x - (1 - (1 - I0) * np.exp(-R0 * x)), 0.0, 1.0, xtol=1e-14)
numerical = final_size("alpha", 0.0)                 # no confinement (alpha = 0)
print(f"SIR final size for R0 = {R0:g}:  exact = {exact:.6f}   numerical = {numerical:.6f}   |diff| = {abs(exact - numerical):.1e}")


## 3. Figure 2 — spontaneous confinement (constant rate $\alpha$)
Solid lines: quasi-steady approximation, Eq. (7). Dotted lines: numerical integration. Insets: $\omega=1.0$ (the approximation improves for fast return). Data first, then the plot.

In [ ]:
# ---------------------------------------------------------------------------
# 3. Figure 2 -- Spontaneous confinement (constant rate alpha): final size rho_r(inf)
#    Solid lines: quasi-steady approximation, Eq. (7).  Dotted lines: numerical integration.
# ---------------------------------------------------------------------------
gamma = PARAMS["gamma"]

# (a) rho_r(inf) versus alpha for beta = 0.5 and 1.0.  Main panel: omega = 0.025.  Inset: omega = 1.0.
alpha_main = np.linspace(0.001, 0.3, 300)
alpha_inset = np.linspace(0.001, 10.0, 200)
fig2a = {}
for beta in (0.5, 1.0):
    fig2a[beta] = dict(
        main=analytic_and_numeric(alpha_main,
                                  lambda a: rho_inf_const(beta, gamma, a, omega=0.025),
                                  lambda a: final_size("alpha", a, t_end=2000, beta=beta, omega=0.025)),
        inset=analytic_and_numeric(alpha_inset,
                                   lambda a: rho_inf_const(beta, gamma, a, omega=1.0),
                                   lambda a: final_size("alpha", a, t_end=2000, beta=beta, omega=1.0)))

# (b) rho_r(inf) versus beta for alpha = 0.01 and 0.1.  Main panel: omega = 0.025.  Inset: omega = 1.0.
beta_main = np.linspace(0.01, 2.0, 300)
beta_inset = np.linspace(0.0, 0.5, 200)
fig2b = {}
for alpha in (0.01, 0.1):
    fig2b[alpha] = dict(
        main=analytic_and_numeric(beta_main,
                                  lambda b: rho_inf_const(b, gamma, alpha, omega=0.025),
                                  lambda b: final_size("alpha", alpha, t_end=6000, beta=b, omega=0.025)),
        inset=analytic_and_numeric(beta_inset,
                                   lambda b: rho_inf_const(b, gamma, alpha, omega=1.0),
                                   lambda b: final_size("alpha", alpha, t_end=6000, beta=b, omega=1.0)))

# (c) rho_r(inf) versus omega for alpha = 0.01, 0.05 and 0.1 (beta = 0.2)
omega_range = np.linspace(0.001, 0.5, 300)
fig2c = {}
for alpha in (0.01, 0.05, 0.1):
    fig2c[alpha] = analytic_and_numeric(omega_range,
                                        lambda w: rho_inf_const(0.2, gamma, alpha, omega=w),
                                        lambda w: final_size("alpha", alpha, t_end=5000, beta=0.2, omega=w))

# (d) phase diagram in the R0-alpha plane: rho_r(inf) from Eq. (7) and the transition line alpha_c, Eq. (9)
R0_range, alpha_range = np.linspace(0.5, 5.0, 200), np.linspace(0.0, 0.15, 200)
heat = np.array([[rho_inf_const(R0 * gamma, gamma, a, omega=0.025) for R0 in R0_range] for a in alpha_range])
alpha_c = np.where(R0_range >= 1.0, (R0_range - 1.0) * 0.025, 0.0)


In [ ]:
# Figure 2: plotting
# ---- figure ----------------------------------------------------------------------------------------
fig = new_fig(W_FULL, 13.6)
L, B, PW, PH, GX, GY = 1.85, 1.5, 5.7, 5.2, 1.8, 1.35          # margins, panel size, gaps [cm]
top_y, bot_y = B + PH + GY, B
axA = axes_cm(fig, L, top_y, PW, PH);            axB = axes_cm(fig, L + PW + GX, top_y, PW, PH)
axC = axes_cm(fig, L, bot_y, PW, PH);            axD = axes_cm(fig, L + PW + GX, bot_y, PW, PH)
c_red, c_blue, c_green = "red", "blue", "green"
f2 = FormatStrFormatter("%.2f")

# (a)
for beta, col in ((0.5, c_red), (1.0, c_blue)):
    plot_pair(axA, alpha_main, *fig2a[beta]["main"], col, label=rf"$\beta={beta}$")
axA.set(xlim=(0, 0.3), ylim=(-0.03, 1.35), xlabel=r"$\alpha$", ylabel=r"$\rho_r^{\infty}$")
axA.set_yticks([0, 0.25, 0.5, 0.75, 1.0]); axA.yaxis.set_major_formatter(f2)
axA.legend(loc="upper right", ncol=1, borderaxespad=0.2)                    # headroom above the data (also hosts the panel label)
ins = axA.inset_axes([0.59, 0.38, 0.38, 0.38])
for beta, col in ((0.5, c_red), (1.0, c_blue)):
    plot_pair(ins, alpha_inset, *fig2a[beta]["inset"], col, lw_ana=2.0, lw_num=1.4)
ins.set(xlim=(0, 10), ylim=(-0.05, 1.05), xticks=[0, 5, 10], yticks=[0, 0.5, 1], yticklabels=["0", "0.5", "1"])
ins.text(0.96, 0.95, r"$\omega=1.0$", transform=ins.transAxes, ha="right", va="top", fontsize=FS_ANNOT)

# (b)
for alpha, col in ((0.01, c_red), (0.1, c_blue)):
    plot_pair(axB, beta_main, *fig2b[alpha]["main"], col, label=rf"$\alpha={alpha}$")
axB.set(xlim=(0, 2), ylim=(-0.03, 1.35), xlabel=r"$\beta$", ylabel=r"$\rho_r^{\infty}$")
axB.set_yticks([0, 0.25, 0.5, 0.75, 1.0]); axB.yaxis.set_major_formatter(f2)
axB.legend(loc="upper right", ncol=1, borderaxespad=0.2)                          # headroom above the data (also hosts the panel label)
ins = axB.inset_axes([0.58, 0.12, 0.37, 0.36])
for alpha, col in ((0.01, c_red), (0.1, c_blue)):
    plot_pair(ins, beta_inset, *fig2b[alpha]["inset"], col, lw_ana=2.0, lw_num=1.4)
ins.set(xlim=(0, 0.5), ylim=(-0.05, 1.05), xticks=[0, 0.25, 0.5], xticklabels=["0", "0.25", "0.5"],
        yticks=[0, 0.5, 1], yticklabels=["0", "0.5", "1"])
ins.text(0.96, 0.08, r"$\omega=1.0$", transform=ins.transAxes, ha="right", va="bottom", fontsize=FS_ANNOT)

# (c)
for alpha, col in ((0.01, c_blue), (0.05, c_red), (0.1, c_green)):
    plot_pair(axC, omega_range, *fig2c[alpha], col, label=rf"$\alpha={alpha}$")
axC.set(xlim=(0, 0.5), ylim=(-0.02, 0.95), xlabel=r"$\omega$", ylabel=r"$\rho_r^{\infty}$")
axC.set_yticks([0, 0.2, 0.4, 0.6, 0.8])
axC.legend(loc="lower right")

# (d)
pcm = axD.pcolormesh(R0_range, alpha_range, heat, cmap="viridis", shading="auto", rasterized=True)
axD.plot(R0_range[R0_range >= 1], alpha_c[R0_range >= 1], color="white", lw=1.6, ls=(0, (4, 2.5)))
axD.text(2.0, 0.105, r"$\mathrm{Healthy\ Phase}$", color="white", fontsize=FS_ANNOT + 1, ha="center", va="center")
axD.text(3.5, 0.016, r"$\mathrm{Epidemic\ Phase}$", color="black", fontsize=FS_ANNOT + 1, ha="center", va="center")
axD.set(xlim=(0.5, 5), ylim=(0, 0.15), xlabel=r"$R_0$", ylabel=r"$\alpha$")
axD.set_xticks([1, 2, 3, 4, 5]); axD.set_yticks([0, 0.05, 0.10, 0.15]); axD.yaxis.set_major_formatter(f2)
axD.tick_params(color="white", which="both", top=False, right=False)
cax = axes_cm(fig, L + PW + GX + PW + 0.2, bot_y, 0.28, PH)
cb = fig.colorbar(pcm, cax=cax); cb.set_ticks([0.2, 0.4, 0.6, 0.8]); cb.ax.tick_params(length=2.5, right=True)
cb.set_label(r"$\rho_r^{\infty}$", fontsize=FS_LABEL)

for ax, letter in zip((axA, axB, axC), "abc"):
    panel_label_inside(ax, letter)
panel_label_inside(axD, "d", color="white")
# value of omega used in panels (a) and (b), written next to the panel label
for ax in (axA, axB):
    ax.text(0.17, 0.95, r"$\omega=0.025$", transform=ax.transAxes, fontsize=FS_ANNOT, ha="left", va="top")
    
save_fig(fig, "fig2")
plt.show()


## 4. Figure 3 — prevalence-driven confinement (rate $\eta\rho_i$)
Panels (a)–(d): final size versus $\eta$, $\beta$ and $\omega$. Damped oscillations in $\eta$ appear for small $\omega$.

In [ ]:
# ---------------------------------------------------------------------------
# 4. Figure 3 -- Prevalence-driven confinement (rate eta * rho_i): final size rho_r(inf)
#    Dotted lines: numerical integration.  Solid lines (panel b): quasi-steady approximation, Eq. (22).
# ---------------------------------------------------------------------------
# (a) rho_r(inf) versus eta for several omega (beta = 0.2, gamma = 0.1). The integration time is very long
#     (t_end = 2.5e6) because for small omega secondary outbreaks can appear at very late times.
eta_log = np.logspace(-1, 2, 200)
eta_lin = np.linspace(0.0, 0.7, 100)                 # inset: zoom on the oscillatory region
omegas_a = [0.0, 0.001, 0.01, 0.025, 0.1]
fig3a = {w: (np.array([final_size("eta", e, t_end=2.5e6, omega=w) for e in eta_log]),
             np.array([final_size("eta", e, t_end=2.5e6, omega=w) for e in eta_lin]))
         for w in omegas_a}

# (b) numerical integration versus the quasi-steady approximation, Eq. (22)
eta_b = np.logspace(-1, 1, 200)
omegas_b = [0.001, 0.01, 0.025]
fig3b = {w: analytic_and_numeric(eta_b,
                                 lambda e: rho_inf_eta(0.2, 0.1, w, e),
                                 lambda e: final_size("eta", e, t_end=3e5, omega=w))
         for w in omegas_b}

# (c) rho_r(inf) versus beta (gamma = 1, eta = 3)
beta_c = np.linspace(0.01, 5, 200)
omegas_c = [0.01, 0.025, 0.1, 0.15]
fig3c = {w: np.array([final_size("eta", 3.0, t_end=5000, omega=w, beta=b, gamma=1.0) for b in beta_c])
         for w in omegas_c}

# (d) rho_r(inf) versus omega for several eta (beta = 0.2, gamma = 0.1)
omega_d = np.logspace(-3, 2, 200)
etas_d = [0.1, 0.2, 0.3, 0.4]
fig3d = {e: np.array([final_size("eta", e, t_end=5e4, omega=w) for w in omega_d]) for e in etas_d}


In [ ]:
# Figure 3: plotting
# Colours. Panels (a), (b) and (c) show curves for different omega: each omega has the same colour in all of them.
omega_color = {0.0: "royalblue", 0.001: "firebrick", 0.01: "darkgreen", 0.025: "darkorange",
               0.1: "darkorchid", 0.15: "teal"}
c_pal = ["royalblue", "firebrick", "darkgreen", "darkorange"]                    # panel (d): one colour per eta
# ---- figure ----------------------------------------------------------------------------------------
fig = new_fig(W_FULL, 14.4)
L, B, PW, PH, GX, GY = 1.85, 1.5, 6.55, 5.6, 1.8, 1.35
top_y, bot_y = B + PH + GY, B
axA = axes_cm(fig, L, top_y, PW, PH);            axB = axes_cm(fig, L + PW + GX, top_y, PW, PH)
axC = axes_cm(fig, L, bot_y, PW, PH);            axD = axes_cm(fig, L + PW + GX, bot_y, PW, PH)
log_ticks = lambda ax, which: getattr(ax, f"set_{which}ticks")([10.0 ** k for k in range(-3, 3)])

def dotted(ax, x, y, color, label=None, lw=1.8):
    ax.plot(x, y, color=color, lw=lw, ls=(0, (1.0, 1.4)), dash_capstyle="round", label=label)

# (a)
for w in omegas_a:
    dotted(axA, eta_log, fig3a[w][0], omega_color[w], label=rf"$\omega={w:g}$")
axA.set(xscale="log", xlim=(0.1, 100), ylim=(-0.0, .9), xlabel=r"$\eta$", ylabel=r"$\rho_r^{\infty}$")
axA.set_yticks([0, 0.2, 0.4, 0.6])
axA.legend(loc="upper right", ncol=1, borderaxespad=0.15)                          # headroom above the data

# (b)
for w in omegas_b:
    plot_pair(axB, eta_b, fig3b[w][0], fig3b[w][1], omega_color[w], label=rf"$\omega={w:g}$", lw_ana=1.5, lw_num=1.5)   # thinner analytical curves so that they do not hide the oscillations
# markers on the numerical curve for omega = 0.01 at eta = 0.18 and eta = 0.25 (valley and peak of the oscillation)
eta_mark = [0.18, 0.25]
rho_mark = [final_size("eta", e, t_end=3e5, omega=0.01) for e in eta_mark]
axB.plot(eta_mark, rho_mark, ls="none", marker="o", ms=5.5, mfc="white", mec=omega_color[0.01], mew=1.5, zorder=5)
axB.set(xlim=(0.1, 1.0), ylim=(0.495, 0.68), xlabel=r"$\eta$", ylabel=r"$\rho_r^{\infty}$")
axB.yaxis.set_major_locator(MultipleLocator(0.05)); axB.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
axB.legend(loc="upper right")

# (c)
for w in omegas_c:
    dotted(axC, beta_c, fig3c[w], omega_color[w], label=rf"$\omega={w:g}$")
axC.set(xlim=(0, 5), ylim=(-0.02, 0.84), xlabel=r"$\beta$", ylabel=r"$\rho_r^{\infty}$")
axC.set_yticks([0, 0.2, 0.4, 0.6, 0.8])
axC.legend(loc="lower right")

# (d)
for e, col in zip(etas_d, c_pal):
    dotted(axD, omega_d, fig3d[e], col, label=rf"$\eta={e:g}$")
axD.set(xscale="log", xlim=(1e-3, 1e2), ylim=(0.49, 0.82), xlabel=r"$\omega$", ylabel=r"$\rho_r^{\infty}$")
axD.set_yticks([0.5, 0.6, 0.7, 0.8])
axD.legend(loc="lower right", ncol=2, columnspacing=1.0, bbox_to_anchor=(1.02, 0.0))
ins = axD.inset_axes([0.62, 0.34, 0.37, 0.42])
for e, col in zip(etas_d, c_pal):
    dotted(ins, omega_d, fig3d[e], col, lw=1.5)
ins.set(xlim=(0.001, 0.015), ylim=(0.50, 0.54), xticks=[0.005, 0.010], xticklabels=["0.005", "0.010"],
        yticks=[0.51, 0.53], yticklabels=["0.51", "0.53"])      # zoom of the same axes (omega, rho_r^inf): no axis labels

for ax, letter in zip((axA, axB, axC, axD), "abcd"):
    panel_label_inside(ax, letter)
save_fig(fig, "fig3")
plt.show()


## 5. Figure 4 — time evolution (secondary outbreaks)
Only $\eta$ changes between columns: $\eta=0.18$ (beginning of the bump: a single wave) and $\eta=0.25$ (maximum of the bump: a second wave) ($\omega=0.01$).

In [ ]:
# ---------------------------------------------------------------------------
# 5. Figure 4 -- Time evolution of the SIRC model with prevalence-driven confinement
#    (beta = 0.2, gamma = 0.1, omega = 0.01) for two values of eta (0.18 and 0.25). Log-spaced time axis.
# ---------------------------------------------------------------------------
beta4, gamma4, omega4 = PARAMS["beta"], PARAMS["gamma"], 0.01
t_log = np.concatenate([[0.0], np.logspace(-2, 5, 2400)])       # t = 0, then 0.01 ... 1e5 (log-spaced)
traj = {eta: time_series("eta", eta, 0.0, np.inf, t_log, omega=omega4) for eta in (0.18, 0.25)}   # always confined


In [ ]:
# Figure 4: plotting
fig = new_fig(W_FULL, 9.7)
L, PW, GX = 1.6, 6.35, 1.6
yb, hb = 1.45, 2.2                                    # bottom panel (rho_i, log scale)
yt, ht = yb + hb + 1.0, 3.9                           # top panel (all compartments)
inset_lims = {0.18: ((1e2, 1e3), (0.36, 0.64)), 0.25: ((1e2, 1e3), (0.36, 0.64))}
col = dict(S="green", C="blue", I="red", R="pink")
axes4 = []
for k, eta in enumerate((0.18, 0.25)):
    x0 = L + k * (PW + GX)
    axT, axB = axes_cm(fig, x0, yt, PW, ht), axes_cm(fig, x0, yb, PW, hb)
    Y = traj[eta]; t = t_log[1:]
    for name, j in (("S", 1), ("C", 0), ("I", 2), ("R", 3)):
        axT.plot(t, Y[1:, j], color=col[name], lw=1.6, label=rf"$\rho_{name}$")
    axT.axhline(gamma4 / beta4, color="magenta", lw=1.1, ls=(0, (4, 2.5)))
    axT.set(xscale="log", xlim=(1, 1e4), ylim=(-0.02, 1.05), yticks=[0, 0.25, 0.5, 0.75, 1.0])
    axT.yaxis.set_major_formatter(FormatStrFormatter("%.2f")); axT.set_xticks([1, 1e1, 1e2, 1e3, 1e4])
    axT.tick_params(labelbottom=False)
    axT.legend(loc="center left", bbox_to_anchor=(0.0, 0.37), ncol=1)
    axT.text(0.03, 0.90, rf"$\omega={omega4:g}$" + "\n" + rf"$\eta={eta:g}$", transform=axT.transAxes, ha="left", va="top",
             fontsize=FS_ANNOT, linespacing=1.3)
    xl, yl = inset_lims[eta]
    ins = axT.inset_axes([0.60, 0.62, 0.34, 0.33])
    ins.plot(t, Y[1:, 1], color=col["S"], lw=1.4)
    ins.plot(t, Y[1:, 3], color=col["R"], lw=1.4)
    ins.axhline(gamma4 / beta4, color="magenta", lw=1.0, ls=(0, (4, 2.5)))
    ins.set(xscale="log", xlim=xl, ylim=yl)
    ins.xaxis.set_major_locator(FixedLocator([1e2, 1e3])); ins.xaxis.set_minor_locator(FixedLocator([]))
    ins.yaxis.set_major_locator(MultipleLocator(0.1)); ins.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    # bottom panel: infected fraction on a log-log scale
    mask = Y[1:, 2] > 0
    axB.plot(t[mask], Y[1:, 2][mask], color="red", lw=1.6, label=r"$\rho_I$")
    axB.set(xscale="log", yscale="log", xlim=(1, 1e4), ylim=(1e-8, 2e-1), xlabel=r"$t$")
    axB.set_xticks([1, 1e1, 1e2, 1e3, 1e4]); axB.set_yticks([1e-8, 1e-6, 1e-4, 1e-2])
    axB.legend(loc="upper left")
    panel_label(axT, "ab"[k])
    axes4.append((axT, axB))
save_fig(fig, "fig4")
plt.show()


## 6. Figure 5 — finite-duration campaign, spontaneous confinement
(a) final size vs start time; (b) dynamics for $t_{ini}<t_{ini}^*$, $\simeq t_{ini}^*$ and $>t_{ini}^*$ ($\alpha=0.05$; the same panel for $\alpha=0.5$ is Supplementary Fig. S2).

In [ ]:
# ---------------------------------------------------------------------------
# 6. Figure 5 -- Finite-duration campaign, spontaneous confinement (Sec. 4.1)
#    Campaign: alpha(t) = alpha_max for t_ini <= t <= min(t_ini + tau, T), 0 otherwise.
#    (a) final size vs campaign start time t_ini;  (b) dynamics for three values of t_ini (alpha = 0.05).
# ---------------------------------------------------------------------------
beta6, gamma6, omega6 = PARAMS["beta"], PARAMS["gamma"], PARAMS["omega"]
tau6, T6, TEND = 50.0, 250.0, 3000.0       # campaign duration, campaign deadline, final integration time

# ---- (a) rho_r(inf) as a function of t_ini ---------------------------------------------------------
t_scan = np.arange(0.0, 300.0 + 1e-9, 0.5)
alpha_ctx = [0.005, 0.01, 0.5]              # context curves
alpha_focus = 0.05                         # curve analysed in panel (b)
R_ref6 = final_size("alpha", 0.0, t_end=TEND)                                  # no campaign
R_ctx6 = {a: np.array([campaign_final_size("alpha", a, t, tau6, T6, TEND) for t in t_scan]) for a in alpha_ctx}
R_focus6 = np.array([campaign_final_size("alpha", alpha_focus, t, tau6, T6, TEND) for t in t_scan])
t_star6 = t_scan[np.argmin(R_focus6)]      # optimal start time
print(f"Fig. 5: no-campaign rho_r = {R_ref6:.4f};  optimal t_ini* = {t_star6:.1f}  (min rho_r = {R_focus6.min():.4f})")

# ---- (b) dynamics for t_ini < t*, ~ t* and > t* ----------------------------------------------------
T_EXT = T6 + 300.0
t_dyn = np.round(np.arange(0.0, T_EXT + 1e-9, 0.1), 6)       # time grid of the dynamics panels

def dynamics(mode, a_max, t_ini, tau, T):
    """Control signal Gamma_{s->c}(t), R_eff(t) and rho_i(t), with a campaign ("ctrl") and without it ("base")."""
    t_off = min(t_ini + tau, T)
    out = {}
    for key, a in (("ctrl", a_max), ("base", 0.0)):
        Y = time_series(mode, a, t_ini, t_off, t_dyn)                       # columns: c, s, i, r
        campaign_on = (t_dyn >= t_ini) & (t_dyn <= t_off) & (a > 0)
        signal = np.where(campaign_on, a * (Y[:, 2] if mode == "eta" else 1.0), 0.0)
        out[key] = dict(sig=signal, Reff=Y[:, 1] * beta6 / gamma6, I=Y[:, 2])
    return out


In [ ]:
FS_TITLE_DYN = 11                                                      # column-title size (3-column figures)
# Figure 5: plotting
X_MAX_DYN = 400                                                         # right end of the time axis in the dynamics panels

def draw_dynamics(fig, mode, amax, t_inits, col_titles, ylabels, ylims, xticks, left, bottom, width, height,
                  letter, sig_color="black", gap_x=0.35, gap_y=0.3, lab_margin=2.25, inside=False, xminor=None):
    """3 x len(t_inits) grid of dynamics panels sharing x in each column; returns the axes."""
    nc, nr = len(t_inits), 3
    pw = (width - lab_margin - (nc - 1) * gap_x) / nc
    ph = (height - 0.75 - (nr - 1) * gap_y - 1.3) / nr            # 0.75 cm for column titles, 1.3 cm for x labels
    axs = np.empty((nr, nc), dtype=object)
    for c, t_ini in enumerate(t_inits):
        d = dynamics(mode, amax, t_ini, tau6, T6)
        x1, x2 = t_ini, min(t_ini + tau6, T6)
        rows = [(d["ctrl"]["sig"], d["base"]["sig"], sig_color, ylims[0]),
                (d["ctrl"]["Reff"], d["base"]["Reff"], "red", ylims[1]),
                (d["ctrl"]["I"], d["base"]["I"], "purple", ylims[2])]
        for r, (y, ybase, color, yl) in enumerate(rows):
            ax = axes_cm(fig, left + lab_margin + c * (pw + gap_x), bottom + (nr - 1 - r) * (ph + gap_y) + 1.3, pw, ph)
            ax.axvspan(x1, x2, color="orange", alpha=0.18, lw=0)
            ax.axvline(x1, color="black", lw=0.8, ls=":", alpha=0.7); ax.axvline(x2, color="black", lw=0.8, ls=":", alpha=0.7)
            ax.axvline(T6, color="purple", lw=1.0, ls=(0, (4, 2.5)), alpha=0.8)
            if r == 1: ax.axhline(1.0, color="darkgray", lw=1.1, ls=(0, (4, 2.5)))
            ax.plot(t_dyn, ybase, color="black", lw=1.5, ls=(0, (1.0, 1.4)), dash_capstyle="round")
            ax.plot(t_dyn, y, color=color, lw=1.6)
            ax.set(xlim=(0, X_MAX_DYN), ylim=yl, xticks=xticks)
            if xminor is not None: ax.set_xticks(xminor, minor=True)
            if c == 0: ax.set_ylabel(ylabels[r], labelpad=2)
            else: ax.tick_params(labelleft=False)
            if r < nr - 1: ax.tick_params(labelbottom=False)
            else: ax.set_xlabel(r"$t$", labelpad=1)
            if r == 0: ax.set_title(col_titles[c], fontsize=FS_TITLE_DYN, pad=5)
            axs[r, c] = ax
    if inside: axs[0, -1].text(0.97, 0.95, rf"$\mathbf{{({letter})}}$", transform=axs[0, -1].transAxes, fontsize=FS_PANEL, ha="right", va="top", zorder=10)      # label inside the top-right panel of the block
    else: fig.text(left / fig.wcm, (bottom + height - 0.05) / fig.hcm, rf"$\mathbf{{({letter})}}$", fontsize=FS_PANEL, va="top")
    return axs

# ---- figure ----------------------------------------------------------------------------------------
fig = new_fig(W_FULL, 17.2)
hb6 = 9.2                                                               # height of the (b) block [cm]
axA = axes_cm(fig, 1.95, hb6 + 1.15 + 1.5, W_FULL - 1.95 - 0.35, 4.7)
ctx_cols = ["steelblue", "#CD9B1D", "#2E8B57"]
CTX_ALPHA = 0.6                                                         # transparency of the context (background) curves
axA.axhline(R_ref6, color="black", lw=1.2, ls=(0, (4, 2.5)), label=r"$\rho_r^{\infty}\ (\alpha=0)$")
for a, c in zip(alpha_ctx, ctx_cols):
    axA.plot(t_scan, R_ctx6[a], color=c, lw=1.7, ls=(0, (1.0, 1.4)), alpha=CTX_ALPHA, dash_capstyle="round", label=rf"$\alpha={a:g}$")
axA.plot(t_scan, R_focus6, color="crimson", lw=2.2, label=rf"$\alpha={alpha_focus:g}\ \mathrm{{(see\ (b))}}$")
T_B5 = [t_star6 - 24.5, t_star6, t_star6 + 15.5]                       # start times of panel (b): t* - 24.5, t*, t* + 15.5
axA.axvline(T_B5[0], color="0.3", lw=0.9, ls=":"); axA.axvline(T_B5[2], color="0.3", lw=0.9, ls=":")
axA.axvline(t_star6, color="crimson", lw=1.1, ls=(0, (4, 2.5)), label=rf"$t_{{ini}}^{{*}}\simeq{t_star6:.1f}$")
axA.set(xlim=(-8, 308), ylim=(0.49, 0.80), xlabel=r"$t_{ini}$", ylabel=r"$\rho_r^{\infty}$")
axA.yaxis.set_major_locator(MultipleLocator(0.1))
axA.xaxis.set_major_locator(MultipleLocator(50)); axA.xaxis.set_minor_locator(MultipleLocator(25)); axA.yaxis.set_minor_locator(MultipleLocator(0.05))   # labels every 50 / 0.1, minor ticks every 25 / 0.05
axA.legend(loc="lower right", ncol=1, bbox_to_anchor=(1.0, 0.02))
axA.text(0.015, 0.04, rf"$\tau={tau6:g}$" + "\n" + rf"$\beta={beta6:g}$" + "\n" + rf"$\gamma={gamma6:g}$" + "\n" + rf"$\omega={omega6:g}$",
         transform=axA.transAxes, ha="left", va="bottom", fontsize=FS_ANNOT, linespacing=1.3)
axA.text(0.985, 0.90, r"$\mathbf{(a)}$", transform=axA.transAxes, fontsize=FS_PANEL, ha="right", va="top", zorder=10)   # (a) inside the panel

draw_dynamics(fig, "alpha", alpha_focus, T_B5,
              [rf"$t_{{\mathrm{{ini}}}}={T_B5[0]:g}<t_{{\mathrm{{ini}}}}^{{*}}$", rf"$t_{{\mathrm{{ini}}}}={T_B5[1]:g}\simeq t_{{\mathrm{{ini}}}}^{{*}}$", rf"$t_{{\mathrm{{ini}}}}={T_B5[2]:g}>t_{{\mathrm{{ini}}}}^{{*}}$"],
              [r"$\Gamma_{s\to c}$", r"$R_{\mathrm{eff}}$", r"$\rho_i$"],
              [(0, 0.1), (0, 2.0), (0, 0.25)], [0, 80, 160, 240, 320], 0.0, 0.0, W_FULL - 0.35, hb6 + 1.15, "b", inside=True,
              xminor=[40, 120, 200, 280, 360])
save_fig(fig, "fig5")
plt.show()


## 7. Figure 6 — optimal start time of the campaign
Heat maps of the final size over the $(\tau, t_{ini})$ plane for the spontaneous (top) and prevalence-driven (bottom) models.

The data of this figure (six matrices of $250\times251$ values, $\tau=1,\dots,250$ in rows and $t_{ini}=0,\dots,250$ in columns) were computed with the Julia script `compute_fig6_data.jl` (Tsit5, `reltol=1e-8`, multithreaded) and are stored in `data_fig_6/fig6_step1_panel1..6.csv`. The next cell loads them and the cell after the warning plots them.

In [ ]:
# ---------------------------------------------------------------------------
# 7. Figure 6 -- load the pre-computed data (Julia, compute_fig6_data.jl)
#    File fig6_step1_panel{k}.csv, k = 1..6, has shape (n_tau, n_tini) = (250, 251): rows tau = 1..250, columns t_ini = 0..250.
#    Panels 1-3: spontaneous confinement alpha = 0.005, 0.01, 0.5.  Panels 4-6: prevalence-driven eta = 0.25, 0.35, 1.0.
# ---------------------------------------------------------------------------
from pathlib import Path

DATA5_DIR = Path("data_fig_6")                      # folder with the CSV files written by compute_fig6_data.jl
STEP5 = 1                                           # grid spacing of the stored data
T5, TEND5 = 250.0, 3000.0                           # campaign deadline T and final integration time
t_grid = np.arange(0.0, 250.0 + 1, 1.0)             # start times t_ini = 0..250 (columns)
tau_grid = np.arange(1.0, 250.0 + 1, 1.0)           # campaign durations tau = 1..250 (rows)

panels5 = [("alpha", a, rf"$\alpha={a:g}$") for a in (0.005, 0.01, 0.5)] + \
          [("eta", e, rf"$\eta={e:g}$") for e in (0.25, 0.35, 1.0)]

files5 = [DATA5_DIR / f"fig6_step1_panel{k}.csv" for k in range(1, 7)]
missing = [f.name for f in files5 if not f.exists()]
if missing:
    raise FileNotFoundError(f"Missing data files in '{DATA5_DIR}/': {missing}. Run compute_fig6_data.jl (Julia) or the optional "
                            "recomputation cell below (RECOMPUTE_FIG6 = True).")
# R5[k][i, j] = final size for duration tau_grid[i] and start time t_grid[j]
R5 = [np.loadtxt(f, delimiter=",") for f in files5]
assert all(R.shape == (len(tau_grid), len(t_grid)) for R in R5), [R.shape for R in R5]

vmin, vmax = min(R.min() for R in R5), max(R.max() for R in R5)
t_opt5 = [t_grid[np.argmin(R, axis=1)] for R in R5]          # optimal start time for each duration
print("Fig. 6: rho_r(inf) range = [%.3f, %.3f]" % (vmin, vmax))
i100 = np.argmin(np.abs(tau_grid - 100))
for (mode, strength, _), t_opt in zip(panels5, t_opt5):
    print(f"  {mode:5s} = {strength:<6g} t_ini*(tau = {tau_grid[0]:g}, 100, 250) = {t_opt[0]:.0f}, {t_opt[i100]:.0f}, {t_opt[-1]:.0f}")


### Optional: recompute the data of Fig. 6 in Python

> **Warning — slow.** The cell below recomputes the six heat maps from scratch with `scipy` (one ODE integration for each $(\tau,t_{ini})$ pair). With the coarse grid `STEP5 = 3` it needs $6\times84\times84\approx4\cdot10^4$ integrations (**roughly 15–30 minutes**); with `STEP5 = 1` (full resolution, $6\times250\times251\approx3.8\cdot10^5$ integrations) it takes **several hours** and is not parallelised. For that reason it is disabled by default (`RECOMPUTE_FIG6 = False`) and the figure uses the stored Julia data. Set `RECOMPUTE_FIG6 = True` to run it; it overwrites `R5`, `t_grid`, `tau_grid` and `STEP5` in memory (it does not modify the CSV files), so the plotting cell below then shows the recomputed data.

In [ ]:
# ---------------------------------------------------------------------------
# 7. Figure 6 -- Final size rho_r(inf) as a function of campaign duration tau and start time t_ini
#    Top row: spontaneous confinement (alpha model).  Bottom row: prevalence-driven (eta model).
#    Cyan line: optimal start time t_ini*(tau).  White dashed line: t_ini = T - tau.
# ---------------------------------------------------------------------------
# SLOW: set RECOMPUTE_FIG6 = True to recompute the data in Python instead of using the stored Julia data (see warning above).
# STEP5 = 3 -> ~4e4 integrations (roughly 15-30 min); STEP5 = 1 -> ~3.8e5 integrations (several hours, estimate).
RECOMPUTE_FIG6 = False

if RECOMPUTE_FIG6:
    STEP5 = 3
    T5, TEND5 = 250.0, 3000.0                           # campaign deadline T and final integration time
    t_grid = np.arange(0.0, 250.0, STEP5)               # start times t_ini
    tau_grid = np.arange(1.0, 250.0 + 1, STEP5)         # campaign durations tau

    panels5 = [("alpha", a, rf"$\alpha={a:g}$") for a in (0.005, 0.01, 0.5)] + \
              [("eta", e, rf"$\eta={e:g}$") for e in (0.25, 0.35, 1.0)]
    # R5[k][i, j] = final size for duration tau_grid[i] and start time t_grid[j]
    R5 = [np.array([[campaign_final_size(mode, strength, t_ini, tau, T5, TEND5) for t_ini in t_grid] for tau in tau_grid])
          for mode, strength, _ in panels5]
    vmin, vmax = min(R.min() for R in R5), max(R.max() for R in R5)
    t_opt5 = [t_grid[np.argmin(R, axis=1)] for R in R5]          # optimal start time for each duration
    print("Fig. 6: rho_r(inf) range = [%.3f, %.3f]" % (vmin, vmax))
    i100 = np.argmin(np.abs(tau_grid - 100))
    for (mode, strength, _), t_opt in zip(panels5, t_opt5):
        print(f"  {mode:5s} = {strength:<6g} t_ini*(tau = {tau_grid[0]:g}, 100, 250) = {t_opt[0]:.0f}, {t_opt[i100]:.0f}, {t_opt[-1]:.0f}")
else:
    print("Recomputation skipped (RECOMPUTE_FIG6 = False): using the data loaded from", DATA5_DIR)


In [ ]:
# Figure 6: plotting (uses R5, t_grid, tau_grid, t_opt5 from the loading cell, or from the optional recomputation)
# ---- figure ----------------------------------------------------------------------------------------
# Landscape panels using the full text width; the colour bar sits on top so that it does not take width.
fig = new_fig(W_FULL, 12.0)
L, PW, PH, GX, GY, B = 2.45, 4.6, 3.55, 0.3, 0.95, 1.5
top_y = B + PH + GY
axes5 = []
for k, ((mode, val, title), R, t_opt) in enumerate(zip(panels5, R5, t_opt5)):
    row, col = divmod(k, 3)
    ax = axes_cm(fig, L + col * (PW + GX), top_y if row == 0 else B, PW, PH)
    pcm = ax.pcolormesh(tau_grid, t_grid, R.T, cmap="inferno", vmin=vmin, vmax=vmax, shading="nearest", rasterized=True)
    ax.plot(tau_grid, t_opt, color="cyan", lw=1.6)
    ax.plot(tau_grid, T5 - tau_grid, color="white", lw=1.2, ls=(0, (3, 2.5)))
    ax.set(xlim=(tau_grid[0] - STEP5 / 2, tau_grid[-1] + STEP5 / 2), ylim=(t_grid[0] - STEP5 / 2, t_grid[-1] + STEP5 / 2),
           xticks=[0, 100, 200], yticks=[0, 100, 200])
    ax.text(0.04, 0.95, title, transform=ax.transAxes, fontsize=FS_ANNOT + 1, ha="left", va="top", color="black", zorder=10)   # alpha / eta value inside the panel, top left
    ax.tick_params(color="white", which="both", top=False, right=False)
    if row == 1: ax.set_xlabel(r"$\tau$")
    else: ax.tick_params(labelbottom=False)
    if col == 0: ax.set_ylabel(r"$t_{ini}$", labelpad=2)
    else: ax.tick_params(labelleft=False)
    ax.text(0.97, 0.95, rf"$\mathbf{{({'abcdef'[k]})}}$", transform=ax.transAxes, fontsize=FS_PANEL,
            ha="right", va="top", color="black", zorder=10)           # light region: black label is readable
    axes5.append(ax)
# row labels
for row, txt in ((0, "spontaneous\nconfinement"), (1, "prevalence-driven\nconfinement")):
    y0 = top_y if row == 0 else B
    fig.text(0.65 / fig.wcm, (y0 + PH / 2) / fig.hcm, txt, rotation=90, fontsize=FS_LABEL - 5, ha="center", va="center", linespacing=1.15)
# horizontal colour bar on top, spanning the three columns
cax = axes_cm(fig, L, top_y + PH + 0.9, 3 * PW + 2 * GX, 0.28)
cb = fig.colorbar(pcm, cax=cax, orientation="horizontal"); cb.set_ticks([0.5, 0.6, 0.7])
cb.ax.xaxis.set_ticks_position("top"); cb.ax.xaxis.set_label_position("top")
cb.ax.tick_params(length=2.5, top=True, bottom=False)
cb.set_label(r"$\rho_r^{\infty}$", fontsize=FS_LABEL, labelpad=3)
save_fig(fig, "fig6")
plt.show()


## 8. Figure 7 — finite-duration campaign, prevalence-driven confinement
(a) final size vs start time (inset: dependence on $\eta_{max}$); (b) dynamics for three start times ($\eta_{max}=2$).

In [ ]:
# ---------------------------------------------------------------------------
# 8. Figure 7 -- Finite-duration campaign, prevalence-driven confinement (Sec. 4.2)
#    Campaign: eta(t) = eta_max for t_ini <= t <= min(t_ini + tau, T), 0 otherwise.
#    (a) final size vs t_ini (inset: dependence on eta_max);  (b) dynamics for four values of t_ini.
#    The parameters (tau6, T6, TEND, ...) and the function `dynamics` come from the Figure 5 cells.
# ---------------------------------------------------------------------------
# ---- (a) rho_r(inf) as a function of t_ini ---------------------------------------------------------
t_scan7 = np.arange(0.0, T6 + 45 + 1e-9, 0.5)
eta_ctx = [0.25, 0.35, 1.0]          # context curves
eta_focus = 2.0                            # curve analysed in panel (b)
R_ref7 = final_size("alpha", 0.0, t_end=TEND)                                  # no campaign
R_ctx7 = {e: np.array([campaign_final_size("eta", e, t, tau6, T6, TEND) for t in t_scan7]) for e in eta_ctx}
R_focus7 = np.array([campaign_final_size("eta", eta_focus, t, tau6, T6, TEND) for t in t_scan7])
t_star7 = t_scan7[np.argmin(R_focus7)]     # optimal start time
print(f"Fig. 7: optimal t_ini* = {t_star7:.1f} (min rho_r = {R_focus7.min():.4f})")

# inset: rho_r(inf) versus eta_max, for the optimal start time and for three fixed start times
eta_inset = np.linspace(0.0, 3.0, 61)
t_search = np.arange(0.0, T6 + 1e-9, 1.0)  # start times scanned to find the optimum for each eta_max
R_opt_inset = np.array([min(campaign_final_size("eta", e, t, tau6, T6, TEND) for t in t_search) for e in eta_inset])
R_fixed_inset = {t: np.array([campaign_final_size("eta", e, t, tau6, T6, TEND) for e in eta_inset])
                 for t in (90.0, 120.0, 160.0)}


In [ ]:
FS_TITLE_DYN = 11
# Figure 7: plotting
# ---- figure ----------------------------------------------------------------------------------------
fig = new_fig(W_FULL, 17.7)
hb7 = 9.2
AW7 = 10.7                                                   # width of panel (a); legends go to its right
axA = axes_cm(fig, 1.95, hb7 + 1.15 + 1.5, AW7, 5.2)
ctx_cols7 = ["steelblue", "#CD9B1D", "#2E8B57", "purple"]
CTX_ALPHA = 0.6                                              # transparency of the context (background) curves
axA.axhline(R_ref7, color="black", lw=1.2, ls=(0, (4, 2.5)), label=r"$\rho_r^{\infty}\ (\eta=0)$")
for e, c in zip(eta_ctx, ctx_cols7):
    axA.plot(t_scan7, R_ctx7[e], color=c, lw=1.7, ls=(0, (1.0, 1.4)), alpha=CTX_ALPHA, dash_capstyle="round", label=rf"$\eta={e:g}$")
axA.plot(t_scan7, R_focus7, color="crimson", lw=2.2, label=rf"$\eta={eta_focus:g}$")
for tv in (90, 160): axA.axvline(tv, color="0.3", lw=0.9, ls=":")
axA.axvline(t_star7, color="crimson", lw=1.1, ls=(0, (4, 2.5)), label=rf"$t_{{ini}}^{{*}}\simeq{t_star7:.1f}$")
axA.set(xlim=(-8, 303), ylim=(0.515, 0.805), xlabel=r"$t_{ini}$", ylabel=r"$\rho_r^{\infty}$")
axA.yaxis.set_major_locator(MultipleLocator(0.1)); axA.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
axA.xaxis.set_major_locator(MultipleLocator(50)); axA.xaxis.set_minor_locator(MultipleLocator(25)); axA.yaxis.set_minor_locator(MultipleLocator(0.05))   # labels every 50 / 0.1, minor ticks every 25 / 0.05
axA.legend(loc="upper left", ncol=1, bbox_to_anchor=(1.02, 1.03))      # legend outside, to the right
axA.text(0.015, 0.04, rf"$\tau={tau6:g},\ \beta={beta6:g}$" + "\n" + rf"$\gamma={gamma6:g},\ \omega={omega6:g}$",
         transform=axA.transAxes, ha="left", va="bottom", fontsize=FS_ANNOT, linespacing=1.3)
ins = axA.inset_axes([0.665, 0.22, 0.30, 0.42])
ins.plot(eta_inset, R_opt_inset, color="royalblue", lw=1.8, label=r"$t_{ini}=t_{ini}^{*}(\eta)$")
for t, c in zip((90.0, 120.0, 160.0), ("firebrick", "darkgreen", "darkorange")):
    ins.plot(eta_inset, R_fixed_inset[t], color=c, lw=1.4, ls=(0, (4, 2)), label=rf"$t_{{ini}}={t:g}$")
ins.set(xlim=(0, 3), ylim=(0.5, 0.8), xlabel=r"$\eta$", ylabel=r"$\rho_r^{\infty}$", xticks=[0, 1, 2, 3], yticks=[0.5, 0.6, 0.7, 0.8])
ins.yaxis.set_major_formatter(FormatStrFormatter("%.1f")); ins.xaxis.labelpad = 1; ins.yaxis.labelpad = 1
ins.legend(loc="upper left", bbox_to_anchor=(1.02, 0.30), bbox_transform=axA.transAxes, fontsize=FS_LEGEND)   # outside, below the main legend
axA.text(0.975, 0.90, r"$\mathbf{(a)}$", transform=axA.transAxes, fontsize=FS_PANEL, ha="right", va="top", zorder=10)   # panel label inside the panel

draw_dynamics(fig, "eta", eta_focus, [90.0, 132.0, 160.0],
              [r"$t_{\mathrm{ini}}=90<t_{\mathrm{ini}}^{*}$", r"$t_{\mathrm{ini}}=132\simeq t_{\mathrm{ini}}^{*}$", r"$t_{\mathrm{ini}}=160>t_{\mathrm{ini}}^{*}$"],
              [r"$\Gamma_{s\to c}$", r"$R_{\mathrm{eff}}$", r"$\rho_i$"],
              [(0, 0.35), (0, 2.0), (0, 0.25)], [0, 80, 160, 240, 320], 0.0, 0.0, W_FULL - 0.35, hb7 + 1.15, "b", inside=True,
              xminor=[40, 120, 200, 280, 360])
save_fig(fig, "fig7")
plt.show()


## 9. Supplementary Figure S1 — no-return case ($\omega=0$), analytical vs. numerical
Final size $\rho_r^{\infty}$ versus $\eta$ when confined agents never return ($\omega=0$), for prevalence-driven confinement. The solid curves solve the transcendental equation of the manuscript (Eq. 3.16),
$$\rho_r^{\infty}=\frac{\beta}{\eta+\beta}\left\{1-\exp\left[-\frac{\eta+\beta}{\gamma}\,\rho_r^{\infty}\right]\right\},$$
and the dotted curves come from the numerical integration of Eqs. 3.11 (the SIRC model with $\omega=0$, $\rho_i(0)=10^{-6}$).

In [ ]:
# ---------------------------------------------------------------------------
# 9. Supplementary Figure S1 -- no-return case (omega = 0): Eq. (3.16) versus numerical integration of Eqs. (3.11)
# ---------------------------------------------------------------------------
beta_S1 = [0.15, 0.2, 0.3, 0.5]            # infection rates (R0 = beta / gamma > 1 in all cases)
gamma_S1 = PARAMS["gamma"]
eta_S1 = np.linspace(0.0, 5.0, 51)
TEND_S1 = 5000.0                           # the epidemic is over long before this time (omega = 0)

def rho_inf_no_return(beta, gamma, eta):
    """Eq. (3.16): non-trivial root of rho = beta/(eta+beta) * (1 - exp(-(eta+beta) rho / gamma))."""
    equation = lambda x: x - beta / (eta + beta) * (1 - np.exp(-(eta + beta) / gamma * x))
    return brentq(equation, 1e-10, 1.0, xtol=1e-14)

S1_ana = {b: np.array([rho_inf_no_return(b, gamma_S1, e) for e in eta_S1]) for b in beta_S1}
S1_num = {b: np.array([final_size("eta", e, t_end=TEND_S1, omega=0.0, beta=b, gamma=gamma_S1) for e in eta_S1]) for b in beta_S1}
max_diff = max(np.max(np.abs(S1_ana[b] - S1_num[b])) for b in beta_S1)
print(f"Fig. S1: max |analytical - numerical| = {max_diff:.1e}")

# ---- figure ----------------------------------------------------------------------------------------
fig = new_fig(8.6, 6.3)
axA = axes_cm(fig, 1.95, 1.45, 6.4, 4.5)
cols_S1 = ["steelblue", "#CD9B1D", "#2E8B57", "crimson"]
for b, c in zip(beta_S1, cols_S1):
    plot_pair(axA, eta_S1, S1_ana[b], S1_num[b], c, label=rf"$\beta={b:g}$")
axA.plot([], [], color="0.3", lw=2.6, alpha=0.5, label=r"$\mathrm{analytical}$")
axA.plot([], [], color="0.3", lw=1.7, ls=(0, (1.0, 1.4)), dash_capstyle="round", label=r"$\mathrm{numerical}$")
axA.set(xlim=(0, 5), ylim=(0, 0.9), xlabel=r"$\eta$", ylabel=r"$\rho_r^{\infty}$")
axA.legend(loc="upper right", ncol=2, columnspacing=0.8, fontsize=FS_LEGEND - 1)
axA.text(0.965, 0.63, rf"$\gamma={gamma_S1:g},\ \omega=0$", transform=axA.transAxes, ha="right", va="top", fontsize=FS_ANNOT)   # parameters below the legend
save_fig(fig, "figS1")
plt.show()


## 9b. Supplementary Figure S2 — Fig. 5(b) for a stronger campaign, $\alpha=0.5$
Same as Fig. 5(b) (spontaneous confinement, $\tau=50$, $T=250$) for $\alpha=0.5$; the corresponding curve of $\rho_r^\infty$ versus $t_{ini}$ is already shown, dotted, in Fig. 5(a). The three start times are $t_{ini}^{*}-24.5$, $t_{ini}^{*}$ and $t_{ini}^{*}+15.5$, where $t_{ini}^{*}$ is the optimal start time found by scanning $t_{ini}$. It uses the parameters and helper functions of the Figure 5 cells.

In [ ]:
FS_TITLE_DYN = 11
# ---------------------------------------------------------------------------
# 9b. Supplementary Figure S2 -- Fig. 5(b) for alpha = 0.5
# ---------------------------------------------------------------------------
alpha_S2 = 0.5
R_S2 = np.array([campaign_final_size("alpha", alpha_S2, t, tau6, T6, TEND) for t in t_scan])
t_star_S2 = t_scan[np.argmin(R_S2)]
print(f"Fig. S2: alpha = {alpha_S2:g}: optimal t_ini* = {t_star_S2:.1f}  (min rho_r = {R_S2.min():.4f})")

tc = [round(t_star_S2 - 24.5), t_star_S2, round(t_star_S2 + 15.5)]
t_inits = [float(t) for t in tc]
dyn = [dynamics("alpha", alpha_S2, t, tau6, T6) for t in t_inits]
sig_max = max(d["ctrl"]["sig"].max() for d in dyn)
i_max = max(d["ctrl"]["I"].max() for d in dyn)
ylims = [(0, np.ceil(1.1 * sig_max / 0.05) * 0.05), (0, 2.0), (0, max(0.25, np.ceil(1.1 * i_max / 0.05) * 0.05))]
titles = [rf"$t_{{\mathrm{{ini}}}}={tc[0]:g}<t_{{\mathrm{{ini}}}}^{{*}}$", rf"$t_{{\mathrm{{ini}}}}={tc[1]:g}\simeq t_{{\mathrm{{ini}}}}^{{*}}$", rf"$t_{{\mathrm{{ini}}}}={tc[2]:g}>t_{{\mathrm{{ini}}}}^{{*}}$"]
# same layout as Supplementary Fig. S3 (sister figure): same height, label with the model parameter at the top left, same time ticks
fig = new_fig(W_FULL, 9.8 + 0.7)
bottom = 0.0
draw_dynamics(fig, "alpha", alpha_S2, t_inits, titles, [r"$\Gamma_{s\to c}$", r"$R_{\mathrm{eff}}$", r"$\rho_i$"],
              ylims, [0, 80, 160, 240, 320], 0.0, bottom, W_FULL - 0.35, 9.8, "a", xminor=[40, 120, 200, 280, 360])
fig.texts[-1].remove()                                      # single block: no panel letter
fig.text(2.3 / fig.wcm, (bottom + 9.8 + 0.1) / fig.hcm, rf"$\alpha={alpha_S2:g}$", fontsize=FS_PANEL, va="bottom")
save_fig(fig, "figS2")
plt.show()


## 10. Supplementary Figure S3 — dynamics of the prevalence-driven campaign for $\eta=0.5$
Same as Fig. 7(b) (control signal $\Gamma_{s\to c}=\eta\rho_i$, effective reproduction number and prevalence, for three start times), but for $\eta=0.5$. The second start time is the optimal one, $t_{ini}^{*}$ (found by scanning $t_{ini}$ with the same campaign as in Fig. 7: $\tau=50$, $T=250$). It uses the parameters and helper functions of the Figure 5/7 cells.

In [ ]:
FS_TITLE_DYN = 11
# ---------------------------------------------------------------------------
# 10. Supplementary Figure S3 -- Fig. 7(b) for eta = 0.5
#     (same campaign as Fig. 7: tau = 50, T = 250; the third column is the optimal start time t_ini*)
# ---------------------------------------------------------------------------
eta_S3 = [0.5]
t_scan_S3 = np.arange(0.0, T6 + 1e-9, 0.5)
t_star_S3 = {}
for e in eta_S3:
    R_scan = np.array([campaign_final_size("eta", e, t, tau6, T6, TEND) for t in t_scan_S3])
    t_star_S3[e] = t_scan_S3[np.argmin(R_scan)]
    print(f"Fig. S3: eta = {e:g}: optimal t_ini* = {t_star_S3[e]:.1f}  (min rho_r = {R_scan.min():.4f})")

fig = new_fig(W_FULL, 9.8 + 0.7)
for n, e in enumerate(eta_S3):
    t_inits = [90.0, float(np.round(t_star_S3[e])), 160.0]
    dyn = [dynamics("eta", e, t, tau6, T6) for t in t_inits]
    sig_max = max(d["ctrl"]["sig"].max() for d in dyn)
    i_max = max(d["ctrl"]["I"].max() for d in dyn)
    ylims = [(0, np.ceil(1.1 * sig_max / 0.05) * 0.05), (0, 2.0), (0, max(0.25, np.ceil(1.1 * i_max / 0.05) * 0.05))]
    titles = [rf"$t_{{\mathrm{{ini}}}}={t_inits[0]:g}<t_{{\mathrm{{ini}}}}^{{*}}$", rf"$t_{{\mathrm{{ini}}}}={t_inits[1]:g}\simeq t_{{\mathrm{{ini}}}}^{{*}}$",
              rf"$t_{{\mathrm{{ini}}}}={t_inits[2]:g}>t_{{\mathrm{{ini}}}}^{{*}}$"]
    bottom = 0.0
    draw_dynamics(fig, "eta", e, t_inits, titles, [r"$\Gamma_{s\to c}$", r"$R_{\mathrm{eff}}$", r"$\rho_i$"],
                  ylims, [0, 80, 160, 240, 320], 0.0, bottom, W_FULL - 0.35, 9.8, "a", xminor=[40, 120, 200, 280, 360])
    fig.texts[-1].remove()                                  # single block: no panel letter
    fig.text(2.3 / fig.wcm, (bottom + 9.8 + 0.1) / fig.hcm, rf"$\eta={e:g}$", fontsize=FS_PANEL, va="bottom")
save_fig(fig, "figS3")
plt.show()


## 11. Output files

In [ ]:
# ---------------------------------------------------------------------------
# 9. Output files
# ---------------------------------------------------------------------------
for f in sorted(os.listdir(OUTDIR)):
    print(f"{OUTDIR}/{f}  ({os.path.getsize(os.path.join(OUTDIR, f)) / 1024:.0f} kB)")
